# 01 · Your first LLM call

**AI Engineering Bootcamp · Module 1**

In this notebook you will:

1. Call an LLM from Python using the OpenAI SDK
2. Read the answer, the token usage and the latency
3. Send a conversation with system, user and assistant messages
4. See what temperature does
5. Switch from Gemini to Groq by changing two lines
6. Compare a weak prompt with a strong prompt

**Before you start**

- Open this folder in VS Code and select your `.venv` as the kernel (top right, *Select Kernel*).
- Copy `.env.example` to `.env` and paste in your own keys.
- Run a cell with **Shift + Enter**.

## Step 0 · Install packages (only needed once)

This installs into the same environment as the notebook's kernel.

In [ ]:
%pip install -q -r requirements.txt

## Step 1 · Load your keys

In [1]:
import os
import time
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # reads the .env file in this folder

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")
GROQ_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")

# Never print the key itself, only whether it was found
print("Gemini key found:", bool(os.getenv("GEMINI_API_KEY")))
print("Groq key found:  ", bool(os.getenv("GROQ_API_KEY")))

Gemini key found: True
Groq key found:   True


## Step 2 · Make your first call

In [2]:
client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url=GROQ_BASE_URL,
)

response = client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[
        {"role": "system", "content": "You are a concise tutor."},
        {"role": "user", "content": "Explain tokens in two lines."},
    ],
)

print(response.choices[0].message.content)

Tokens are the smallest pieces of text (words or sub‑words) the model processes; each token carries a chunk of meaning and counts toward the input/output limit.  
The model predicts the next token step by step, building responses from these discrete units.


**If you get an error**

| Error | Usual cause |
|---|---|
| 401 | Wrong or missing key. Check `.env` and re-run Step 1. |
| 404 | Wrong model name. Copy the current name from the provider's model list. |
| 429 | Rate limit. Wait a minute, or switch to Groq (Step 6). |

## Step 3 · Read the response: text, tokens and time

In [ ]:
start = time.perf_counter()
response = client.chat.completions.create(
    model=GEMINI_MODEL,
    messages=[{"role": "user", "content": "Give me 3 tips for writing clean Python."}],
)
latency = time.perf_counter() - start

print(response.choices[0].message.content)
print("-" * 40)
print("model:", response.model)
usage = response.usage
if usage:
    print("tokens in: ", usage.prompt_tokens)
    print("tokens out:", usage.completion_tokens)
print(f"latency:    {latency:.2f}s")

## Step 4 · Messages and roles

The model has **no memory between calls**. A conversation is just a list of messages you send every time.
Here we keep the list, add the model's reply, then ask a follow-up question.

In [ ]:
messages = [
    {"role": "system", "content": "You are a friendly Python tutor. Keep answers short."},
    {"role": "user", "content": "What is a list in Python?"},
]

reply = client.chat.completions.create(model=GEMINI_MODEL, messages=messages)
answer = reply.choices[0].message.content
print("Assistant:", answer)

# Add the reply and a follow-up question, then send the whole list again
messages.append({"role": "assistant", "content": answer})
messages.append({"role": "user", "content": "Show me an example."})

reply = client.chat.completions.create(model=GEMINI_MODEL, messages=messages)
print("\nAssistant:", reply.choices[0].message.content)
print("\nMessages sent this time:", len(messages))

## Step 5 · Temperature

Lower temperature gives more focused, repeatable answers. Higher temperature gives more varied ones.

In [ ]:
question = [{"role": "user", "content": "Suggest a name for a study-planner app. Reply with the name only."}]

for temp in [0.2, 0.2, 0.9, 0.9]:
    r = client.chat.completions.create(model=GEMINI_MODEL, messages=question, temperature=temp)
    print(f"temperature {temp}: {r.choices[0].message.content.strip()}")

## Step 6 · Switch provider: Gemini to Groq

Same code. Only the key, `base_url` and model change.

In [ ]:
groq_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url=GROQ_BASE_URL,
)

start = time.perf_counter()
r = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[{"role": "user", "content": "Explain tokens in two lines."}],
)
print(r.choices[0].message.content)
print(f"\nGroq latency: {time.perf_counter() - start:.2f}s")

## Step 7 · Weak prompt vs strong prompt

In [ ]:
text = """Git is a version control system. It tracks changes to files over time,
lets several people work on the same project, and makes it possible to go back
to an earlier version when something breaks."""

weak = "Summarize this."
strong = (
    "Summarize the text below in 3 bullet points for a beginner. "
    "Use simple English. If the text is not about software, reply 'Off topic'."
)

for name, prompt in [("WEAK", weak), ("STRONG", strong)]:
    r = client.chat.completions.create(
        model=GEMINI_MODEL,
        messages=[{"role": "user", "content": f"{prompt}\n\n{text}"}],
    )
    print(f"--- {name} ---")
    print(r.choices[0].message.content, "\n")

## Try it yourself

1. Change the system message in Step 2 to *"You explain everything using cricket examples."* What changes?
2. In Step 4, ask a third follow-up question. How many tokens in does it use now, and why?
3. In Step 7, replace `text` with a paragraph that is **not** about software. Does the strong prompt say *Off topic*?